In [20]:
import os
import numpy as np
import torch
import torch.nn as nn
import tensorflow as tf
from tensorflow import keras
from torch.utils.data import DataLoader, TensorDataset
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from PIL import Image
import kagglehub

In [21]:
path = kagglehub.dataset_download("harishkumardatalab/food-image-classification-dataset")

path_to_food = path + '/Food Classification dataset'
categories = os.listdir(path_to_food)
print(f"Количество категорий: {len(categories)}")

data_x, data_y = [], []
size = (120, 120)

Количество категорий: 34


In [22]:
for i, cat in enumerate(categories):
    folder = os.path.join(path_to_food, cat)
    files = os.listdir(folder)[:100] 
    for f in files:
        img = Image.open(os.path.join(folder, f)).convert('RGB').resize(size)
        data_x.append(np.array(img) / 255.0) 
        data_y.append(i) 

In [23]:
data_x = np.array(data_x, dtype='float32')
data_y = np.array(data_y, dtype='int64')
print(f"Количество: {data_x.shape[0]} , размер: {data_x.shape[1:]}")

Количество: 3400 , размер: (120, 120, 3)


In [24]:
x_train, x_test, y_train, y_test = train_test_split(data_x, data_y, test_size=0.15, random_state=42)
print(f"Количество на train:{len(x_train)} , Количество на test: {len(x_test)} фото")

Количество на train:2890 , Количество на test: 510 фото


In [25]:
#TF
model_tf = keras.Sequential([
    keras.Input(shape=(120, 120, 3)),
    keras.layers.Conv2D(32, (3,3), activation='relu'),
    keras.layers.MaxPooling2D(2,2),
    keras.layers.Conv2D(64, (3,3), activation='relu'),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(34, activation='softmax')
])

model_tf.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [26]:
model_tf.fit(x_train, y_train, epochs=10, batch_size=32, verbose=0)
_, acc_tf = model_tf.evaluate(x_test, y_test, verbose=0)
print(f"Точность TF: {acc_tf:.2%}")

Точность TF: 13.33%


In [27]:
#PYTORCH
transform = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

In [28]:
x_train_pt = torch.tensor(x_train).permute(0, 3, 1, 2).float()
x_train_pt = transform(x_train_pt)
y_train_pt = torch.tensor(y_train).long()

x_test_pt = torch.tensor(x_test).permute(0, 3, 1, 2).float()
x_test_pt = transform(x_test_pt)
y_test_pt = torch.tensor(y_test).long()

train_loader = DataLoader(TensorDataset(x_train_pt, y_train_pt), batch_size=32, shuffle=True)

print(f"Размер: {x_train_pt.shape[1:]}")

Размер: torch.Size([3, 120, 120])


In [29]:
model_pt = nn.Sequential(
    nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2,2),
    nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2,2),
    nn.Flatten(),
    nn.Linear(64 * 28 * 28, 128), nn.ReLU(),
    nn.Linear(128, 34)
)

opt_pt = torch.optim.Adam(model_pt.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [30]:
for epoch in range(10):
    for batch_x, batch_y in train_loader:
        opt_pt.zero_grad()
        loss = criterion(model_pt(batch_x), batch_y)
        loss.backward()
        opt_pt.step()

with torch.no_grad():
    pt_preds = model_pt(x_test_pt)
    acc_pt = (pt_preds.argmax(1) == y_test_pt).float().mean().item()
print(f"Точность PyTorch: {acc_pt:.2%}")

Точность PyTorch: 23.33%


In [31]:
#EfficientNet
model_eff = models.efficientnet_b0(weights='DEFAULT')

In [32]:
for param in model_eff.parameters():
    param.requires_grad = False

In [33]:
in_features = model_eff.classifier[1].in_features 

model_eff.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, 34)
)

In [34]:
opt_eff = torch.optim.Adam(model_eff.classifier.parameters(), lr=0.001)

In [35]:
for epoch in range(10):
    for batch_x, batch_y in train_loader:
        opt_eff.zero_grad()
        loss = criterion(model_eff(batch_x), batch_y)
        loss.backward()
        opt_eff.step()

with torch.no_grad():
    eff_preds = model_eff(x_test_pt)
    acc_eff = (eff_preds.argmax(1) == y_test_pt).float().mean().item()
print(f"Точность EfficientNet: {acc_eff:.2%}")

Точность EfficientNet: 51.18%


In [36]:
with torch.no_grad():
    eff_preds = model_eff(x_test_pt)
    acc_eff = (eff_preds.argmax(1) == y_test_pt).float().mean().item()
print(f"Точность EfficientNet: {acc_eff:.2%}")

Точность EfficientNet: 50.20%


In [37]:
print(f"1. TF:        {acc_tf:.2%}")
print(f"2. PT:        {acc_pt:.2%}")
print(f"3. EfficientNet:    {acc_eff:.2%}")

1. TF:        13.33%
2. PT:        23.33%
3. EfficientNet:    50.20%
